# 05 — CST Far-Field Import

Synthesise a CST-format far-field text file (matching the column layout that  
CST Studio exports), load it with `beamlab21.cst.load_cst`, inspect the  
Cartesian projection, and stack multiple single-frequency exports into one cube.

**No external data needed** — the CST text files are generated entirely from a  
synthetic Gaussian beam.

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from beamlab21.cst import load_cst, stack

## 1 · Helper — write a fake CST `.txt` export

CST far-field exports have two header rows, then columns:  
`Theta  Phi  Abs(E)  Abs(Cross)  Phase(Cross)  Abs(Copol)  Phase(Copol)  Ax.Ratio`

In [ ]:
def write_cst_txt(path, freq_mhz, sigx=9.0, sigy=8.0, xo=0.2, yo=-0.1,
                  theta_max=75.0, n_theta=76, n_phi=360):
    """Write a synthetic CST far-field text file at ``path``."""
    thetas = np.linspace(0, theta_max, n_theta)   # deg
    phis   = np.linspace(0, 359, n_phi)            # deg
    T, P   = np.meshgrid(thetas, phis)
    T, P   = T.ravel(), P.ravel()

    # Project to Cartesian, evaluate a Gaussian
    px = T * np.cos(np.deg2rad(P))
    py = T * np.sin(np.deg2rad(P))
    amp_lin = np.exp(
        -((px - xo) ** 2 / (2 * sigx ** 2) + (py - yo) ** 2 / (2 * sigy ** 2))
    )

    # Convert to dB (floor at −60 dB)
    amp_dB = 20.0 * np.log10(np.clip(amp_lin, 1e-3, None))

    # CST columns: Theta Phi Abs(E) Abs(Cross) Phase(Cross) Abs(Copol) Phase(Copol) Ax.Ratio
    data = np.column_stack([
        T,           # col 0 — Theta
        P,           # col 1 — Phi
        amp_dB,      # col 2 — Abs(E)  [total]
        amp_dB - 20, # col 3 — Abs(Cross)
        np.zeros_like(T),  # col 4 — Phase(Cross)
        amp_dB,      # col 5 — Abs(Copol)
        np.zeros_like(T),  # col 6 — Phase(Copol)
        np.zeros_like(T),  # col 7 — Ax.Ratio
    ])

    header = (
        f"Exported far-field data  Frequency={freq_mhz}MHz\n"
        "Theta[deg]  Phi[deg]  Abs(E)[dBV/m]  Abs(Cross)[dBV/m]  "
        "Phase(Cross)[deg]  Abs(Copol)[dBV/m]  Phase(Copol)[deg]  Ax.Ratio[dB]"
    )
    np.savetxt(path, data, header=header, comments="", fmt="%.6f")

print("write_cst_txt() defined")

## 2 · Write and load a single-frequency file

In [ ]:
FREQ_MHZ = 400.0
SIGX, SIGY = 9.0, 8.0

_tmp = tempfile.NamedTemporaryFile(suffix=".txt", delete=False)
_tmp.close()
cst_path = _tmp.name

write_cst_txt(cst_path, FREQ_MHZ, sigx=SIGX, sigy=SIGY)
print(f"Synthetic CST file → {cst_path}")

x, y, freq_arr, data = load_cst(
    cst_path, freq_mhz=FREQ_MHZ,
    column="copol", size=301, xy_max=30.0,
)
print(f"\nLoaded  →  grid {data.shape}  freq {freq_arr*1e3} MHz")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

beam2d = data[0]
beam2d_norm = beam2d / beam2d.max()

im = axes[0].pcolormesh(x, y, beam2d_norm, cmap="inferno", vmin=0, vmax=1)
plt.colorbar(im, ax=axes[0], label="Normalised amplitude")
axes[0].set_title(f"Cartesian beam from CST  ({FREQ_MHZ:.0f} MHz)")
axes[0].set_xlabel("x [deg]")
axes[0].set_ylabel("y [deg]")

mid = len(x) // 2
axes[1].plot(x, beam2d_norm[mid, :], label="EW cut (y=0)")
axes[1].plot(y, beam2d_norm[:, mid], label="NS cut (x=0)", linestyle="--")
axes[1].axhline(0.5, color="k", linewidth=0.7, linestyle=":")
axes[1].set_xlabel("Angle [deg]")
axes[1].set_ylabel("Normalised amplitude")
axes[1].set_title("1D cuts")
axes[1].legend()

plt.tight_layout()
plt.show()

Path(cst_path).unlink(missing_ok=True)

## 3 · Multi-frequency stacking with `cst.stack`

In [ ]:
FREQS = [300.0, 400.0, 500.0, 600.0]   # MHz
tmp_files = []

for f in FREQS:
    # Beam narrows with frequency: σ ∝ 1/ν  (λ/D scaling)
    scale = FREQ_MHZ / f
    t = tempfile.NamedTemporaryFile(suffix=".txt", delete=False)
    t.close()
    write_cst_txt(t.name, f, sigx=SIGX * scale, sigy=SIGY * scale)
    tmp_files.append(t.name)

x_s, y_s, freq_s, cube = stack(
    tmp_files, FREQS, column="copol", size=201, xy_max=30.0,
)
print(f"Stacked cube shape : {cube.shape}")
print(f"Frequencies [GHz]  : {freq_s}")

for f in tmp_files:
    Path(f).unlink(missing_ok=True)

In [ ]:
fig, axes = plt.subplots(1, len(FREQS), figsize=(14, 3.5))
mid_s = len(x_s) // 2

for ax, freq_ghz, chan in zip(axes, freq_s, range(len(FREQS))):
    img = cube[chan]
    ax.pcolormesh(x_s, y_s, img / img.max(), cmap="inferno", vmin=0, vmax=1)
    ax.set_title(f"{freq_ghz*1e3:.0f} MHz")
    ax.set_xlabel("x [deg]")
    ax.set_ylabel("y [deg]")

plt.suptitle("Multi-frequency CST cube (beam narrows with ν)", y=1.02)
plt.tight_layout()
plt.show()

## 4 · Column options: `copol`, `e`, `cross`

In [ ]:
t2 = tempfile.NamedTemporaryFile(suffix=".txt", delete=False)
t2.close()
write_cst_txt(t2.name, 400.0)

cols = ["copol", "e", "cross"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col in zip(axes, cols):
    _, _, _, d = load_cst(t2.name, 400.0, column=col, size=201, xy_max=30.0)
    ax.pcolormesh(x_s, y_s, d[0] / d[0].max(), cmap="inferno", vmin=0, vmax=1)
    ax.set_title(f"column='{col}'")
    ax.set_xlabel("x [deg]")
    ax.set_ylabel("y [deg]")

plt.tight_layout()
plt.show()
Path(t2.name).unlink(missing_ok=True)